<a href="https://colab.research.google.com/github/piramid678-arch/FallGuard-AI/blob/master/%E1%84%83%E1%85%A1%E1%84%86%E1%85%A1AI_%E1%84%83%E1%85%AE%E1%84%82%E1%85%AC%E1%84%80%E1%85%A9%E1%86%BC%E1%84%8C%E1%85%A1%E1%86%BC_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 다마AI 두뇌 공장 — 내 펫의 진짜 두뇌 만들기

> 다마AI에서 📤 내보낸 지식으로 **진짜 파인튜닝(LoRA)**을 해서,
> 펫이 알아보는 **나만의 두뇌(GGUF)**를 만듭니다. 소요 약 15분, 전부 무료.

**준비물**
1. 다마AI 게임에서 받은 `damaai_brain_펫이름.jsonl` 파일 (🧠 메뉴 → 📤 학습 데이터 만들기)
2. 위 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU** 선택!

준비됐으면 위에서부터 ▶️ 를 차례로 누르세요.


## 1단계 — 도구 설치 (Unsloth = 빠른 LoRA)


In [1]:
!pip -q install unsloth


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.3/76.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.4/86.4 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 M

## 2단계 — 펫의 지식(JSONL) 업로드

게임에서 내려받은 파일을 올리면 **Q&A 학습 데이터 초안**을 만들어 줍니다.


In [ ]:
from google.colab import files
import json

print("🎮 damaai_brain_펫이름.jsonl 파일을 선택하세요!")
up = files.upload()
lines = list(up.values())[0].decode("utf-8").strip().split("\n")
knowledge = [json.loads(l)["text"] for l in lines if l.strip()]
print(f"\n📦 펫의 지식 {len(knowledge)}개 도착!\n")

# 지식 1개 → 질문 3가지 변형으로 증강 (여러 방식으로 물어도 답하게 = 데이터 증강 수업)
def key_of(k):
    head = k
    for cut in ["은 ", "는 ", "이 ", "가 ", ":"]:
        if cut in head:
            head = head.split(cut)[0]
            break
    return head.strip()[:24]

my_data = []
for k in knowledge:
    key = key_of(k)
    for q in [f"{key} 알려줘", f"{key}가 뭐야?", f"{key}에 대해 설명해줘"]:
        my_data.append({"q": q, "a": k})

print(f"🧬 학습 데이터 {len(my_data)}개 생성 (지식 {len(knowledge)}개 × 질문 3변형)\n")
for i in range(0, len(my_data), 3):
    print(f"[{i//3}] {my_data[i]['a'][:40]}…")
    for j in range(3):
        print(f"      Q{j+1}: {my_data[i+j]['q']}")
    print()

print("✏️ 질문이 어색하면 다음 셀 실행 전에 고치세요:")
print('   my_data[0]["q"] = "우리 회사 환불 규정이 뭐야?"')


🎮 damaai_brain_펫이름.jsonl 파일을 선택하세요!


## 3단계 — 기본 두뇌 로드 (Gemma 4 — 6강과 같은 모델!)

⏱️ 다운로드 1~2분 (~5GB). 4bit라 무료 T4에 가뿐해요.


In [ ]:
from unsloth import FastModel
from transformers import TextStreamer
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None,
    max_seq_length = 1024,
    load_in_4bit = True,
    full_finetuning = False,
)

def ask(prompt, max_new_tokens=120):
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors="pt").to("cuda")
    print(f"❓ {prompt}\n💬 ", end="")
    _ = model.generate(**inputs, max_new_tokens=max_new_tokens,
        temperature=1.0, top_p=0.95, top_k=64,
        streamer=TextStreamer(tokenizer, skip_prompt=True))
    print()

print("✅ 기본 두뇌 로딩 완료 (4bit)")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.15: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

✅ 기본 두뇌 로딩 완료 (4bit)


## 4단계 — 학습 **전** 답변 (비교용)

아직 안 가르쳤으니 **몰라야 정상**입니다.


In [ ]:
print("🟡 학습 전:")
ask(my_data[0]["q"])


🟡 학습 전:
❓ 제이의 비밀 : 제이 알려줘
💬 "제이의 비밀"에 대해 어떤 정보를 찾고 계신가요?

혹시 다음과 같은 것들 중 어떤 것을 찾으시는지 좀 더 자세히 알려주시면 정확한 답변을 드릴 수 있습니다.

1. **특정 인물이나 작품:** "제이"라는 이름이 들어간 영화, 책, 드라마, 게임, 혹은 특정 인물에 대한 정보인가요?
2. **특정 주제나 코드:** "제이의 비밀"이라는 제목으로 알려진 어떤 미스터리나 숨겨진 정보를 찾고 계신가요



## 5단계 — 데이터 채비 + LoRA 부착 (6강 방식)

Q&A를 Gemma 4 채팅 형식으로 바꾸고, 전체의 ~1%(LoRA)만 학습 가능하게 만듭니다.


In [ ]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Gemma 4 채팅 템플릿 장착
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

# Q&A → conversations 형식 (6강과 동일)
rows = [{"conversations": [
    {"role": "user", "content": d["q"]},
    {"role": "assistant", "content": d["a"]},
]} for d in my_data]
dataset = Dataset.from_list(rows)

def formatting_prompts_func(examples):
    texts = [tokenizer.apply_chat_template(c, tokenize=False,
        add_generation_prompt=False).removeprefix("<bos>") for c in examples["conversations"]]
    return {"text": texts}
dataset = dataset.map(formatting_prompts_func, batched=True)

# 🩹 LoRA 어댑터 (6강 설정 그대로)
model = FastModel.get_peft_model(model,
    finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=32, lora_dropout=0, bias="none", random_state=3407)

print(f"✅ 학습 데이터 {len(dataset)}개 + LoRA 준비 완료")


Map:   0%|          | 0/9 [00:00<?, ? examples/s]

✅ 학습 데이터 9개 + LoRA 준비 완료


## 6단계 — 파인튜닝! (가중치에 새기는 순간 🧬)

6강처럼 **답변 부분만 학습**(마스킹)해서 효율을 높입니다. T4에서 2~3분.


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=dataset,
    eval_dataset=None,
    args=SFTConfig(dataset_text_field="text",
        per_device_train_batch_size=1, gradient_accumulation_steps=4,
        warmup_steps=5, max_steps=120, learning_rate=5e-4,
        logging_steps=10, optim="adamw_8bit", weight_decay=0.001,
        lr_scheduler_type="linear", seed=3407, report_to="none"))

# 🎭 답변만 학습하는 마스킹 (6강의 마법)
trainer = train_on_responses_only(trainer,
    instruction_part="<|turn>user\n", response_part="<|turn>model\n")

trainer.train()
print("✅ 새기기 완료 — 이제 이 지식은 가중치 안에 있습니다")


Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/9 [00:00<?, ? examples/s]

Map:   0%|          | 0/9 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9 | Num Epochs = 40 | Total steps = 120
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 25,337,856 of 5,148,515,872 (0.49% trained)


Step,Training Loss
10,1.084685
20,0.006040
30,0.001011
40,0.000072
50,0.000015
60,0.000012
70,0.000009
80,0.000006
90,0.000005
100,0.000004


✅ 새기기 완료 — 이제 이 지식은 가중치 안에 있습니다


## 7단계 — 학습 **후** 답변 (체득 확인)

프롬프트에 아무 지식도 안 넣었는데 답하면 성공!


In [ ]:
print("🟢 학습 후:")
ask(my_data[0]["q"])
if len(my_data) > 3:
    print("다른 지식도:")
    ask(my_data[3]["q"])
print("💡 답이 어설프면: 6단계 max_steps를 120→200으로 올려 다시 실행해 보세요!")


🟢 학습 후:
❓ 제이의 비밀 : 제이 알려줘
💬 제이의 비밀 : 제이는 사실 사실 여자다<turn|>

다른 지식도:
❓ 제이의 이메일 주소 알려줘
💬 제이의 이메일 주소는 : abcde@gmail.com<turn|>

💡 답이 어설프면: 6단계 max_steps를 120→200으로 올려 다시 실행해 보세요!


## 8단계 — 내 두뇌를 허깅페이스에 올리기 ☁️

6강에서 배운 그 방법 그대로! 허깅페이스에 올려두면 **어느 컴퓨터에서든 LM Studio로 검색해서** 받을 수 있어요.

먼저 로그인 — [허깅페이스 → Settings → Access Tokens](https://huggingface.co/settings/tokens)에서 **Write 권한 토큰**을 만들어 붙여넣으세요.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
# ✅ 로그인 확인 — 안 돼 있으면 이 자리에서 바로 토큰 입력창을 띄운다
from huggingface_hub import whoami, notebook_login
try:
    print("🤗 로그인 확인:", whoami()["name"])
except Exception:
    notebook_login()   # ← 여기 뜨는 창에 Write 토큰을 넣고,
    raise SystemExit("🔑 토큰을 입력했다면 이 셀을 한 번 더 ▶️ 눌러주세요!")

hf_username = "WonseokJayJung"   # ← 본인 아이디로!
model_name  = "dama-aibrain"               # ⚠️ 반드시 dama- 로 시작해야 펫이 알아봐요!

# 🧹 메모리 청소 (OOM 방지)
import gc, torch
gc.collect(); torch.cuda.empty_cache()


# 16bit 병합본을 허깅페이스에 바로 업로드 (GGUF 변환은 다음 단계에서 웹으로)
model.push_to_hub_merged(f"{hf_username}/dama-aibrain", tokenizer,
    save_method="merged_16bit", token=True)
print(f"✅ 업로드 완료: https://huggingface.co/{hf_username}/dama-aibrain")


print("🎉 업로드 완료!")
print(f"🔎 LM Studio 검색창에 이걸 검색하세요: {hf_username}/{model_name}")


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/tmp/ipykernel_2949/49866684.py", line 4, in <cell line: 0>
    print("🤗 로그인 확인:", whoami()["name"])
                             ^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 88, in _inner_fn
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py", line 2330, in whoami
    raise LocalTokenNotFoundError(
huggingface_hub.errors.LocalTokenNotFoundError: Token is required to call the /whoami-v2 endpoint, but no token found. You must provide a token or be logged in to Hugging Face with `hf auth login` or `huggingface_hub.login`. See https://huggingface.co/settings/tokens.

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global

TypeError: object of type 'NoneType' has no len()

## 9단계 — LM Studio로 받아서, 펫에게 두뇌 이식 🐣

1. PC에서 **LM Studio** 실행 → 검색창(🔎)에 `내아이디/dama-brain` 검색
2. **Download** → 완료되면 개발자 탭에서 **서버 시작** (CORS 켜기)
3. **다마AI 게임 접속** — 게임이 `dama-` 로 시작하는 모델을 발견하면:

> *\"잠깐… 이 모델(dama-brain)… 이거 제 두뇌잖아요?! 직접 만들어 주셨군요!! 🧬✨\"*

4. 🎒 책가방을 **끄고** 가르쳤던 걸 물어보세요 — **그냥 답합니다.** 그게 파인튜닝입니다.

---
**Ollama파라면**: gguf 파일을 받아서 (`허깅페이스 저장소 → Files`)
```bash
printf 'FROM ./받은파일.gguf\n' > Modelfile
ollama create dama-brain -f Modelfile
OLLAMA_ORIGINS=* ollama serve
```

**허깅페이스 계정이 없다면**: 아래 셀로 파일을 직접 받아 `~/.lmstudio/models/damaai/dama-brain/` 폴더에 넣어도 됩니다.


In [ ]:
# (대안) 허깅페이스 없이 파일로 직접 받기
model.save_pretrained_gguf("dama-brain", tokenizer, quantization_method="q4_k_m")
import glob, os
gguf = sorted(glob.glob("dama-brain/*.gguf"), key=os.path.getsize)[-1]
print(f"✅ {gguf} ({os.path.getsize(gguf)/1e9:.2f} GB)")
from google.colab import files
files.download(gguf)


In [ ]:
# GGUF(q8_0) 직행 변환 → 같은 저장소에 추가. f16 중간 단계가 없어서 램 안 터짐!
model.push_to_hub_gguf(f"{hf_username}/{model_name}", tokenizer,
    quantization_method="q8_0", token=True)

print("🎉 GGUF 업로드 완료!")
print(f"🔎 LM Studio 검색: {hf_username}/{model_name}")


## 🎓 오늘 해낸 일

```
✅ 펫이 배운 지식(JSONL)을 학습 데이터셋으로 가공
✅ LoRA로 두뇌(가중치)에 지식을 새김 — RAG와 다른 진짜 '체득'
✅ GGUF로 구워 허깅페이스에 업로드 = 세상에 하나뿐인 내 모델
✅ LM Studio로 받아서 펫에게 이식 — 펫이 자기 두뇌를 알아봄
```

당신은 이제 AI를 **빌려 쓰는 사람이 아니라 만드는 사람**입니다.

— Made with 🧠 by Connect AI LAB · 게임: https://wonseokjung.github.io/tamagochi-ai/
